In [1]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel
from langchain_ollama import ChatOllama

C:\Users\graj5\AppData\Local\Temp\ipykernel_21724\1056550615.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


## LCEL 1 --Extracting the skill from the resume

In [2]:
sample_resume=PyPDFLoader('Gokul_Resume_Final.pdf').load()[0].page_content
print(sample_resume)

GOKUL RAJ V 
Salem, Tamil Nadu | +91 9944998210 | linkedin.com/in/gokul-raj-2b8796280 |  
CAREER OBJECTIVE 
 
B.Tech graduate in Information Technology with a strong foundation in Python, SQL, Power BI, Machine Learning, and 
Generative AI. Hands-on project experience building a deep learning-based face anti-spoofing system and an interactive Power BI 
sales dashboard. Seeking an entry-level Data Analyst or ML Engineer role to apply analytical and technical skills to real-world, 
data-driven problems. 
TECHNICAL SKILLS 
 
Programming: Python, SQL, OOP 
Data Analytics: Power BI, DAX, Excel, Data Cleaning, Data Visualization 
Machine Learning: Scikit-learn, Regression, Classification, Feature Engineering, Model Evaluation 
Generative AI: Prompt Engineering, LLM Fundamentals 
Libraries: Pandas, NumPy , Matplotlib, Seaborn 
Databases: MySQL, Oracle SQL 
Tools: Jupyter Notebook, VS Code, Git, GitHub 
EDUCATION 
 
B.Tech, Information Technology – Ganesh College of Engineering, Salem (2022 – 

In [3]:
skill_prompt=ChatPromptTemplate.from_messages([
    ('system','extract the skill from the resume.'),
    ('human','{resume}')
])

### -->LLm

In [4]:
llm=ChatOllama(model="qwen2.5:1.5b",temperature=0)

### -->parser

In [5]:
parser=StrOutputParser()

In [6]:
skill_chain=skill_prompt | llm | parser

## LCEL 2-- Summarize the about the project/exp from the resume

In [7]:
summarize_prompt=ChatPromptTemplate.from_messages([
    ('system','summarize the experience from the resume.'),
    ('human','resume :{resume}')
])
summarize_chain=summarize_prompt | llm |parser

## LCEL -Find the missing skill from the resume for data science role

In [8]:
missing_skill=ChatPromptTemplate.from_messages([
    ('system','For data science role find the missing skill from the resume.'),
    ('human','resume : {resume}')
])
missing_skill_chain=missing_skill | llm | parser

In [9]:
final_chain=RunnableParallel(
    skill=skill_chain,
    sumarize=summarize_chain,
    missing_skill=missing_skill_chain
)

In [10]:
result=final_chain.invoke({'resume':sample_resume})
print(result)#skill,summarize,missing_skill

{'skill': 'The skills extracted from the resume are:\n\n1. Programming: Python, SQL, OOP\n2. Data Analytics: Power BI, DAX, Excel, Data Cleaning, Data Visualization\n3. Machine Learning: Scikit-learn, Regression, Classification, Feature Engineering, Model Evaluation\n4. Generative AI: Prompt Engineering, LLM Fundamentals\n5. Libraries: Pandas, NumPy, Matplotlib, Seaborn\n6. Databases: MySQL, Oracle SQL\n7. Tools: Jupyter Notebook, VS Code, Git, GitHub\n\nThe resume also highlights the following skills:\n\n1. Android Development\n2. Face Anti-Spoofing System Using Deep Learning\n3. Sales & Customer Analytics Dashboard\n4. Python Programming Intern\n5. Android Development Intern\n\nThe resume indicates a strong foundation in Python, SQL, and machine learning, as well as experience in data analytics, generative AI, and programming languages such as Python, SQL, and OOP. The candidate has also demonstrated hands-on project experience in building a deep learning-based face anti-spoofing sys

In [11]:
print(result['sumarize'])

Gokul Raj V is a B.Tech graduate in Information Technology with a strong foundation in Python, SQL, Power BI, Machine Learning, and Generative AI. He has hands-on project experience building a deep learning-based face anti-spoofing system and an interactive Power BI sales dashboard. He is seeking an entry-level Data Analyst or ML Engineer role to apply analytical and technical skills to real-world, data-driven problems. His technical skills include programming in Python, SQL, OOP, data analytics in Power BI, DAX, Excel, data cleaning, data visualization, machine learning with Scikit-learn, regression, classification, feature engineering, model evaluation, generative AI with prompt engineering, LLM fundamentals, libraries in Pandas, NumPy, Matplotlib, Seaborn, databases in MySQL, Oracle SQL, tools in Jupyter Notebook, VS Code, Git, GitHub. His education includes B.Tech in Information Technology from Ganesh College of Engineering, Salem (2022 – 2026) with a CGPA of 7.85/10.0 and Higher S